In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df = pd.read_csv('hotel_data_modified.csv')
df.head()

is_canceled --> 0 / 1

- Customers with babies or children may have higher cancellation probability  
- Longer lead time may increase cancellation likelihood  
- Repeated guests may have lower cancellation probability  
- Higher previous cancellations may indicate higher future cancellation risk  
- Longer waiting list days may increase cancellation likelihood  

* What if free room upgrades are offered?  
* If full/half meal plans reduce cancellation rates, consider offering them as paid options

In [ ]:
df.isna().sum()

In [ ]:
df.dtypes

In [ ]:
df['is_canceled'].value_counts()

### 1) First Analysis

In [ ]:
df['children'].unique()

In [ ]:
# Data Cleaning: Handling Missing Values and Converting Data Types (float to int)
df_kids = df.dropna(subset=['children']).copy()
df_kids['children'] = df_kids['children'].astype(int)

In [ ]:
# Outlier Removal
df_kids = df_kids[df_kids['children'] <= 3]
df_kids

# Unrealistic values in the "kids" variable (e.g., 10 children) were removed.
# Only observations with 3 or fewer children were retained for analysis.

In [ ]:
df_kids['children'].value_counts().sort_index()

In [ ]:
# How Does the Number of Children Affect Cancellation Probability?
df_kids.groupby('children')['is_canceled'].mean()

In [ ]:
import statsmodels.api as sm

X = df_kids[['children']]
X = sm.add_constant(X)
y = df_kids['is_canceled']

model = sm.Logit(y, X).fit()
model.summary()

In [ ]:
# coef = 0.0248
# p-value = 0.098

# The results suggest that the number of children is positively associated with cancellation probability.  
# However, this relationship is not statistically significant.

In [ ]:
df_baby = df_kids.copy()
df_baby['babies'].value_counts().sort_index()

In [ ]:
# Outlier Removal
df_baby = df_baby[df_baby['babies'] <= 2]
df_baby

# Unrealistic values in the "babies" variable (e.g., 9 or more babies) were removed. 
# Only observations with 2 or fewer babies were retained for analysis.

In [ ]:
df_baby['babies'].value_counts().sort_index()

In [ ]:
# How Does the Number of Babies Affect Cancellation Probability?
df_baby.groupby('babies')['is_canceled'].mean()

In [ ]:
import statsmodels.api as sm

X = df_baby[['babies']]
X = sm.add_constant(X)
y = df_baby['is_canceled']

model = sm.Logit(y, X).fit()
model.summary()

In [ ]:
# coef = -0.9578
# p-value = 0

# The number of babies has a statistically significant effect on cancellation probability.  
# As the number of babies increases, the likelihood of cancellation decreases.

In [ ]:
# babies + children
df_all = df_kids.copy()
df_all = df_all[df_all['babies'] <= 2]
df_all = df_all[df_all['children'] <= 3]

In [ ]:
import statsmodels.api as sm

X = df_all[['children', 'babies']]
X = sm.add_constant(X)
y = df_all['is_canceled']

model_all = sm.Logit(y, X).fit()
model_all.summary()

In [ ]:
# children coef = 0.0296, p-value = 0.048
# babies coef = -0.9615, p-value = 0.000

# Controlling for the number of babies, the number of children has a statistically significant effect on cancellation probability.  
# As the number of children increases, the likelihood of cancellation also increases.  
# Meanwhile, the number of babies remains a key factor that decreases cancellation probability.

In [ ]:
df_seg = df_kids.copy()

def segment(row):
    if row['babies'] >= 1:
        return 'babies'
    elif row['children'] >= 1:
        return 'children_only'
    else:
        return 'no_kids'

df_seg['segment'] = df_seg.apply(segment, axis=1)

In [ ]:
# Cancellation Rate by Segment
df_seg.groupby('segment')['is_canceled'].mean()

In [ ]:
import matplotlib.pyplot as plt

df_seg.groupby('segment')['is_canceled'].mean().plot(kind='bar')
plt.title('Cancellation Rate by Customer Segment')
plt.ylabel('Cancellation Rate')
plt.show()

## 1. Customer Segmentation

### 1) Customers with Babies (babies >= 1)
- Lower cancellation rate  
- More planned trips  

### 2) Customers with Children (children >= 1, no babies)
- Relatively higher cancellation rate  
- Higher schedule uncertainty  

### 3) General Customers (no kids)
- Average cancellation rate  
- Baseline group  

## 2. Marketing Strategy

### 1) Customers with Babies → Revenue Maximization Strategy
- Paid room upgrades  
- Meal package upselling  
- Baby-friendly premium packages  

### 2) Customers with Children → Cancellation Prevention Strategy
- Shorten or restrict free cancellation period  
- Offer confirmed benefits instead of discounts (e.g., free breakfast if booking is maintained)  

### 3) General Customers
- Apply seasonal and demand-based pricing  
- General promotions (discounts, packages)  
- Encourage add-ons (breakfast, room upgrade)

### 2) Second Analysis

In [ ]:
df['lead_time'].describe()

In [ ]:
df['lead_group'] = pd.cut(df['lead_time'],
                         bins=[0, 30, 90, 180, 365, 800],
                         labels=['very_short', 'short', 'mid', 'long', 'very_long'])

In [ ]:
df.groupby('lead_group', observed=False)['is_canceled'].mean()

In [ ]:
import statsmodels.api as sm

X = df[['lead_time']]
X = sm.add_constant(X)
y = df['is_canceled']

model = sm.Logit(y, X).fit()
model.summary()

In [ ]:
# coef = 0.0059
# p-value = 0

# Lead time has a statistically significant positive effect on cancellation probability.  
# As lead time increases, the likelihood of cancellation also increases.

## Long Lead-Time Customers → Cancellation Prevention Strategy

### Strategy
- Send reminder messages before check-in  
- Offer incentives for maintaining reservations (e.g., free breakfast)  
- Adjust partial refund policies  

**Encouraging customers to reconsider before canceling is important.**

## 3) Third Analysis

In [ ]:
df['is_repeated_guest'].value_counts()

In [ ]:
df.groupby('is_repeated_guest')['is_canceled'].mean()

## 1. Customer Segmentation

### 1) Returning Customers (1)
- Lower cancellation rate  
- High customer loyalty  

### 2) New Customers (0)
- Higher cancellation rate  

## 2. Marketing Strategy

### 1) Retention and Loyalty Building
- VIP / membership programs  
- Reward points  
- Personalized benefits  

### 2) Optimizing First-Time Experience
- Encourage confirmed bookings through incentives  
- Emphasize reviews and trust signals  
- Offer first-time customer discounts  

## 4) Fourth Analysis

In [ ]:
df['previous_cancellations'].value_counts().sort_index()

In [ ]:
def cancel_group(x):
    if x == 0:
        return '0'
    elif x == 1:
        return '1'
    else:
        return '2+'

df['cancel_group'] = df['previous_cancellations'].apply(cancel_group)

In [ ]:
df.groupby('cancel_group')['is_canceled'].mean()

Customers with previous cancellation history show a significantly higher likelihood of canceling.  
Notably, even customers with a single prior cancellation exhibit high cancellation rates.  

- Apply prepayment or stricter cancellation policies to customers with prior cancellations  
- Treat customers without cancellation history as general customers and apply standard promotions  

**Note: The small sample size of customers with two or more prior cancellations may lead to variability in the results.**

## 5) Fifth Analysis

In [ ]:
df['days_in_waiting_list'].value_counts().sort_index()

In [ ]:
df['waiting_group'] = df['days_in_waiting_list'].apply(lambda x: '0' if x == 0 else '1+')

In [ ]:
df.groupby('waiting_group')['is_canceled'].mean()

### Customers on the Waiting List (1+)
- Encourage faster booking confirmation  
- Provide incentives during the waiting period (e.g., discounts, upgrade offers)  
- Send continuous reminder messages  

**It is important to retain customers during the waiting period and prevent drop-off.**

## Evaluating the Effectiveness of Marketing Strategies

In [ ]:
df.groupby('meal')['is_canceled'].mean()

Moderately priced meal-inclusive packages reduce customer burden and increase booking confirmation rates.  
In contrast, full board packages show relatively higher cancellation rates.  

This may be due to higher cost 부담 or characteristics of customers who choose such packages.  
Therefore, to improve booking retention, it is important to design offerings centered around reasonably priced service bundles.


For full board packages, additional conditions or incentives may be needed to increase booking confirmation.

In [ ]:
df['is_upgraded'] = (df['assigned_room_type'] != df['reserved_room_type']).astype(int)
df.groupby('is_upgraded')['is_canceled'].mean()

Analysis based on room upgrade status shows that customers who received upgrades had a very low cancellation rate of approximately 5%, while those who did not receive upgrades showed a significantly higher cancellation rate of about 41%.  

- Offer upgrade benefits to customers with a high risk of cancellation  
- Promote the possibility of upgrades in advance to encourage booking retention  
- Provide paid upgrade options for low-risk customers  

In [ ]:
import statsmodels.api as sm

# Select only the variables needed for analysis
df_model = df[['is_canceled',
               'lead_time',
               'is_repeated_guest',
               'previous_cancellations',
               'children',
               'babies',
               'days_in_waiting_list']].copy()

# Create waiting variable
df_model['waiting'] = (df_model['days_in_waiting_list'] > 0).astype(int)

# Remove missing values
df_model = df_model.dropna()

# Define X and y
X = df_model[['lead_time',
              'is_repeated_guest',
              'previous_cancellations',
              'children',
              'babies',
              'waiting']]

y = df_model['is_canceled']

# Add constant term (intercept)
X = sm.add_constant(X)

# Logistic regression
model = sm.Logit(y, X).fit()
model.summary()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Extract coefficients
coef = model.params

# Remove intercept
coef = coef.drop('const')

# Calculate importance based on absolute values
importance = coef.abs().sort_values(ascending=False)

# Visualization
plt.figure()
importance.plot(kind='bar')

plt.title('Feature Importance (Logistic Regression)')
plt.xlabel('Variables')
plt.ylabel('Absolute Coefficient')

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X = df_model[['lead_time','is_repeated_guest','previous_cancellations','children','babies','waiting']]
y = df_model['is_canceled']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

pred = model.predict(X_test)
accuracy_score(y_test, pred)

In [ ]:
from sklearn.metrics import roc_auc_score
proba = model.predict_proba(X_test)[:,1]
roc_auc_score(y_test, proba)

In [ ]:
result = pd.DataFrame({
    'actual': y_test,
    'prob': proba
})

result.sort_values('prob', ascending=False).head(20)

In [ ]:
high_risk = result[result['prob'] >= 0.7]
len(high_risk) / len(result)

In [ ]:
# 1. Overall Cancellation Rate
overall = result['actual'].mean()

# 2. High-Risk Cancellation Rate
high = high_risk['actual'].mean()

# 3. Comparison
overall, high

## Conclusion

This analysis was conducted to reduce hotel booking cancellation rates.

First, key variables related to booking cancellations were identified through univariate analysis.  
Based on this, lead time, repeat customer status (is_repeated_guest), previous cancellations, number of children (children, babies), and waiting status were selected as key variables.

A multiple logistic regression analysis was then performed to compare the relative impact of these variables.  
The results showed that previous cancellations and repeat customer status were the most influential factors.

Based on these findings, a model was developed to predict cancellation probability for each customer.  
Using this model, approximately 6.7% of customers were identified as high-risk.  
The actual cancellation rate of this group was about 85.1%, which is more than twice the overall average cancellation rate (36.7%).

This indicates that the model effectively identifies customers with a high likelihood of cancellation.

Based on these results, applying proactive strategies to high-risk customers (with a predicted cancellation probability above 70%)—such as requiring prepayment, strengthening booking confirmation, and offering additional incentives—can effectively reduce cancellation rates.

For example, if even a portion of the 1,603 high-risk customers can be retained, approximately 300–400 cancellations could be prevented.  
This would lead to a meaningful reduction in the overall cancellation rate.

In addition, for customers with low cancellation risk, upselling and package offerings can be used to generate additional revenue.

However, this analysis is based on a limited set of variables and does not account for factors such as pricing, schedule changes, or external conditions.  
Future research incorporating these factors could further improve model performance.

Overall, this analysis not only identifies key drivers of booking cancellations but also demonstrates the potential for proactively identifying high-risk customers and implementing targeted strategies to reduce cancellations.